In [1]:
# Test imports
import sys
print(f"Python version: {sys.version}")

import pandas as pd
import numpy as np
import tensorflow as tf
import sklearn

print(f"✅ Pandas version: {pd.__version__}")
print(f"✅ NumPy version: {np.__version__}")
print(f"✅ TensorFlow version: {tf.__version__}")
print(f"✅ Scikit-learn version: {sklearn.__version__}")
print("\n🎉 All packages installed successfully!")

Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
✅ Pandas version: 2.2.3
✅ NumPy version: 2.1.3
✅ TensorFlow version: 2.21.0
✅ Scikit-learn version: 1.6.1

🎉 All packages installed successfully!


In [2]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split, StratifiedKFold
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional, Input
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
print("Libraries imported")

Libraries imported


In [4]:

import pandas as pd

fake = pd.read_excel("Fake.xlsx")
true = pd.read_excel("True.xlsx")

# Keep only the columns needed for this project
required_columns = ["title", "text", "subject", "date"]

fake = fake[required_columns].copy()
true = true[required_columns].copy()

print("Fake articles:", len(fake))
print("Real articles:", len(true))
print("Fake columns:", fake.columns.tolist())
print("Real columns:", true.columns.tolist())

display(fake.head(3))
display(true.head(3))


Fake articles: 23502
Real articles: 21417
Fake columns: ['title', 'text', 'subject', 'date']
Real columns: ['title', 'text', 'subject', 'date']


,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,2017-12-31 00:00:00
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,2017-12-31 00:00:00
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,2017-12-30 00:00:00


,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,2017-12-31
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,2017-12-29
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,2017-12-31


In [ ]:
fake['label'] = 0
true['label'] = 1
df = pd.concat([fake, true], ignore_index=True)
print(f"Total articles: {len(df)}")
print(f"Fake: {len(df[df['label']==0])}, Real: {len(df[df['label']==1])}")

Total articles: 44898
Fake: 23481, Real: 21417


3.Clean text data

In [ ]:
import re
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text
df['cleaned_text'] = df['text'].apply(clean_text)
print("Text cleaning complete")
print(df['cleaned_text'].iloc[0][:200])

Text cleaning complete
donald trump just couldn t wish all americans a happy new year and leave it at that instead he had to give a shout out to his enemies haters and the very dishonest fake news media the former reality s


4.Split data

In [ ]:
from sklearn.model_selection import train_test_split
X = df['cleaned_text']
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

Train: 35918, Test: 8980


5: Tokenization and Padding

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(X_train)
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)
max_len = 500
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')
print(f"Train shape: {X_train_pad.shape}, Test shape: {X_test_pad.shape}")

Train shape: (35918, 500), Test shape: (8980, 500)


In [ ]:
kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
X_cv, y_cv = X_train_pad, y_train.values
lstm_scores, bilstm_scores = [], []

for fold, (train_idx, val_idx) in enumerate(kfold.split(X_cv, y_cv), 1):
    print(f"\nFOLD {fold}/3")
    X_tr, X_val = X_cv[train_idx], X_cv[val_idx]
    y_tr, y_val = y_cv[train_idx], y_cv[val_idx]

    # LSTM
    model = Sequential([Input(shape=(500,)), Embedding(10000, 128),
                       LSTM(100, dropout=0.2), Dropout(0.5), Dense(1, 'sigmoid')])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    model.fit(X_tr, y_tr, epochs=2, batch_size=64, verbose=0)
    lstm_scores.append(model.evaluate(X_val, y_val, verbose=0)[1])

    # BiLSTM
    model_bi = Sequential([Input(shape=(500,)), Embedding(10000, 128),
                          Bidirectional(LSTM(100, dropout=0.2)), Dropout(0.5), Dense(1, 'sigmoid')])
    model_bi.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    model_bi.fit(X_tr, y_tr, epochs=2, batch_size=64, verbose=0)
    bilstm_scores.append(model_bi.evaluate(X_val, y_val, verbose=0)[1])

    print(f"LSTM: {lstm_scores[-1]:.4f}, BiLSTM: {bilstm_scores[-1]:.4f}")

print(f"\nLSTM Mean: {np.mean(lstm_scores)*100:.2f}%")
print(f"BiLSTM Mean: {np.mean(bilstm_scores)*100:.2f}%")


FOLD 1/3
LSTM: 0.8762, BiLSTM: 0.9896

FOLD 2/3
LSTM: 0.9313, BiLSTM: 0.9926

FOLD 3/3
LSTM: 0.9382, BiLSTM: 0.9941

LSTM Mean: 91.52%
BiLSTM Mean: 99.21%


In [ ]:
print("Training final models...")
lstm_final = Sequential([Input(shape=(500,)), Embedding(10000, 128),
                        LSTM(100, dropout=0.2), Dropout(0.5), Dense(1, 'sigmoid')])
lstm_final.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
lstm_final.fit(X_train_pad, y_train, epochs=3, batch_size=64, validation_data=(X_test_pad, y_test), verbose=1)



Training final models...
Epoch 1/3
562/562 ━━━━━━━━━━━━━━━━━━━━ 353s 622ms/step - accuracy: 0.7894 - loss: 0.4250 - val_accuracy: 0.8844 - val_loss: 0.3143
Epoch 2/3
562/562 ━━━━━━━━━━━━━━━━━━━━ 352s 626ms/step - accuracy: 0.7860 - loss: 0.4124 - val_accuracy: 0.9097 - val_loss: 0.3133
Epoch 3/3
562/562 ━━━━━━━━━━━━━━━━━━━━ 376s 668ms/step - accuracy: 0.9421 - loss: 0.2003 - val_accuracy: 0.9298 - val_loss: 0.2424


In [ ]:

bilstm_final = Sequential([Input(shape=(500,)), Embedding(10000, 128),
                          Bidirectional(LSTM(100, dropout=0.2)), Dropout(0.5), Dense(1, 'sigmoid')])
bilstm_final.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
bilstm_final.fit(X_train_pad, y_train, epochs=3, batch_size=64, validation_data=(X_test_pad, y_test), verbose=1)

Epoch 1/3
562/562 ━━━━━━━━━━━━━━━━━━━━ 1217s 2s/step - accuracy: 0.9695 - loss: 0.0810 - val_accuracy: 0.9813 - val_loss: 0.0693
Epoch 2/3
562/562 ━━━━━━━━━━━━━━━━━━━━ 1217s 2s/step - accuracy: 0.9933 - loss: 0.0198 - val_accuracy: 0.9869 - val_loss: 0.0357
Epoch 3/3
562/562 ━━━━━━━━━━━━━━━━━━━━ 1251s 2s/step - accuracy: 0.9971 - loss: 0.0103 - val_accuracy: 0.9962 - val_loss: 0.0137


In [ ]:

print(f"LSTM Test: {lstm_final.evaluate(X_test_pad, y_test, verbose=0)[1]*100:.2f}%")
print(f"BiLSTM Test: {bilstm_final.evaluate(X_test_pad, y_test, verbose=0)[1]*100:.2f}%")

LSTM Test: 92.98%
BiLSTM Test: 99.62%


In [ ]:
# Saving models
lstm_final.save('lstm_final_model.h5')
bilstm_final.save('bilstm_final_model.h5')

# summary
results = {
    'lstm_cv_mean': 91.52,
    'bilstm_cv_mean': 99.21,
    'lstm_test_acc': 92.98,
    'bilstm_test_acc': 99.62
}
import pickle
with open('final_results.pkl', 'wb') as f:
    pickle.dump(results, f)
print("All models and results saved!")

All models and results saved!


In [ ]:
# Compare train vs test accuracy
#overfitting test
train_acc = bilstm_final.evaluate(X_train_pad, y_train, verbose=0)[1]
test_acc = bilstm_final.evaluate(X_test_pad, y_test, verbose=0)[1]

print(f"Train accuracy: {train_acc*100:.2f}%")
print(f"Test accuracy:  {test_acc*100:.2f}%")
print(f"Difference:     {abs(train_acc-test_acc)*100:.2f}%")

if abs(train_acc - test_acc) < 0.03:  # Less than 3% difference
    print("✅ No overfitting detected")
else:
    print(" Possible overfitting - check regularization")

Train accuracy: 99.92%
Test accuracy:  99.62%
Difference:     0.30%
✅ No overfitting detected


In [ ]:
import re
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Define prediction function
def predict_news(text):
    cleaned = re.sub(r'[^a-z\s]', '', text.lower())
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    seq = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(seq, maxlen=500, padding='post')
    prob = bilstm_final.predict(padded, verbose=0)[0][0]

    if prob > 0.5:
        return f"REAL NEWS (confidence: {prob*100:.2f}%)"
    else:
        return f"FAKE NEWS (confidence: {(1-prob)*100:.2f}%)"

# Test with 2 examples
print("Example 1:", predict_news("BREAKING: Obama caught in shocking scandal! MUST READ!"))
print("Example 2:", predict_news("WASHINGTON (Reuters) - The president announced new economic policies today."))

Example 1: FAKE NEWS (confidence: 100.00%)
Example 2: REAL NEWS (confidence: 99.33%)


In [ ]:

# 10 test news articles
test_news = [
    # FAKE NEWS examples (sensational, emotional)
    "BREAKING: Obama caught in shocking scandal with Russian documents! MUST READ before media deletes!",
    "Scientists discover that lemon juice CURES cancer in 3 days! Big Pharma hiding this miracle cure!",
    "Famous actor found dead in hotel room! Police refuse to release details! Mainstream media won't report!",
    "NASA confirms aliens exist but government is hiding truth! Whistleblower reveals secret UFO files!",
    "Election was stolen! Thousands of fraudulent votes discovered! Voting machines were tampered with!",

    # REAL NEWS examples (formal, source-cited)
    "WASHINGTON (Reuters) - The Federal Reserve announced a 0.25% interest rate hike today citing strong economic growth.",
    "According to the World Health Organization, global COVID-19 cases have decreased by 15% this week.",
    "President addressed the nation tonight outlining his new infrastructure plan worth $1 trillion.",
    "Researchers at Stanford University published a study in Nature Medicine showing promising drug results.",
    "The United Nations Security Council passed a resolution calling for humanitarian aid access to conflict zones."
]

print("TESTING 10 NEWS ARTICLES")


for i, news in enumerate(test_news, 1):
    result = predict_news(news)
    print(f"{i}. {news[:60]}...")
    print(f"   → {result}")
    print()

TESTING 10 NEWS ARTICLES
1. BREAKING: Obama caught in shocking scandal with Russian docu...
   → FAKE NEWS (confidence: 100.00%)

2. Scientists discover that lemon juice CURES cancer in 3 days!...
   → FAKE NEWS (confidence: 100.00%)

3. Famous actor found dead in hotel room! Police refuse to rele...
   → FAKE NEWS (confidence: 100.00%)

4. NASA confirms aliens exist but government is hiding truth! W...
   → FAKE NEWS (confidence: 100.00%)

5. Election was stolen! Thousands of fraudulent votes discovere...
   → FAKE NEWS (confidence: 100.00%)

6. WASHINGTON (Reuters) - The Federal Reserve announced a 0.25%...
   → REAL NEWS (confidence: 99.57%)

7. According to the World Health Organization, global COVID-19 ...
   → FAKE NEWS (confidence: 100.00%)

8. President addressed the nation tonight outlining his new inf...
   → FAKE NEWS (confidence: 100.00%)

9. Researchers at Stanford University published a study in Natu...
   → FAKE NEWS (confidence: 100.00%)

10. The United Nations Security

In [ ]:
import pickle

with open("C:\\Users\\DELL\\Downloads\\tokenizer_bilstm_project.pickle", "rb") as f:
    tokenizer = pickle.load(f)

print(type(tokenizer))
print(tokenizer.num_words)
print(len(tokenizer.word_index))

print(list(tokenizer.word_index.items())[:10])

<class 'keras.src.legacy.preprocessing.text.Tokenizer'>
10000
190170
[('the', 1), ('to', 2), ('of', 3), ('a', 4), ('and', 5), ('in', 6), ('that', 7), ('on', 8), ('s', 9), ('for', 10)]


In [ ]:
print(predict_news(
"The Federal Reserve announced a new interest rate decision after reviewing inflation data and economic conditions."
))

print(predict_news(
"Scientists discovered a secret pill that cures every disease instantly and governments are hiding it."
))

NameError: name 'predict_news' is not defined